In [10]:
import lxml.etree as ET
import math
import re
import html
import json

# --- 1. Accent Stripping Function ---

GREEK_ACCENT_MAP = {
    'ά': 'α', 'ὰ': 'α', 'ᾶ': 'α', 'ἀ': 'α', 'ἄ': 'α', 'ἂ': 'α', 'ἆ': 'α', 'ἁ': 'α', 'ἅ': 'α', 'ἇ': 'α', 'ᾀ': 'α',
    'ᾁ': 'α', 'ᾂ': 'α', 'ᾃ': 'α', 'ᾄ': 'α', 'ᾅ': 'α', 'ᾆ': 'α', 'ᾇ': 'α', 'ᾲ': 'α', 'ᾴ': 'α', 'ᾶ': 'α', 'ᾷ': 'α',
    'Ά': 'Α', 'Ἀ': 'Α', 'Ἄ': 'Α', 'Ἂ': 'Α', 'Ἆ': 'Α', 'Ἁ': 'Α', 'Ἅ': 'Α', 'Ἇ': 'Α', 'ᾈ': 'Α', 'ᾉ': 'Α', 'ᾊ': 'Α',
    'ᾋ': 'Α', 'ᾌ': 'Α', 'ᾍ': 'Α', 'ᾎ': 'Α', 'ᾏ': 'Α', 'Ὰ': 'Α', 'Ά': 'Α', 'Ᾰ': 'Α', 'Ᾱ': 'Α', 'ᾼ': 'Α',
    'έ': 'ε', 'ὲ': 'ε', 'ἐ': 'ε', 'ἔ': 'ε', 'ἒ': 'ε', 'ἑ': 'ε', 'ἕ': 'ε',
    'Έ': 'Ε', 'Ἐ': 'Ε', 'Ἔ': 'Ε', 'Ἒ': 'Ε', 'Ἑ': 'Ε', 'Ἕ': 'Ε', 'Ὲ': 'Ε', 'Έ': 'Ε',
    'ή': 'η', 'ὴ': 'η', 'ῆ': 'η', 'ἠ': 'η', 'ἤ': 'η', ' ἢ': 'η', 'ἦ': 'η', 'ἡ': 'η', 'ἥ': 'η', 'ἧ': 'η', 'ᾐ': 'η',
    'ᾑ': 'η', 'ᾒ': 'η', 'ᾓ': 'η', 'ᾔ': 'η', 'ᾕ': 'η', 'ᾖ': 'η', 'ᾗ': 'η', 'ῂ': 'η', 'ῄ': 'η', 'ῆ': 'η', 'ῇ': 'η',
    'Ή': 'Η', 'Ἠ': 'Η', 'Ἤ': 'Η', 'Ἢ': 'Η', 'Ἦ': 'Η', 'Ἡ': 'Η', 'Ἥ': 'Η', 'Ἧ': 'Η', 'ᾘ': 'Η', 'ᾙ': 'Η', 'ᾚ': 'Η',
    'ᾛ': 'Η', 'ᾜ': 'Η', 'ᾝ': 'Η', 'ᾞ': 'Η', 'ᾟ': 'Η', 'Ὴ': 'Η', 'Ή': 'Η', 'ῌ': 'Η',
    'ί': 'ι', 'ὶ': 'ι', 'ῖ': 'ι', 'ἰ': 'ι', 'ἴ': 'ι', 'ἲ': 'ι', 'ἶ': 'ι', 'ἱ': 'ι', 'ἵ': 'ι', 'ἷ': 'ι', 'ῒ': 'ι', 'ΐ': 'ι', 'ῖ': 'ι',
    'Ί': 'Ι', 'Ἰ': 'Ι', 'Ἴ': 'Ι', 'Ἲ': 'Ι', 'Ἶ': 'Ι', 'Ἱ': 'Ι', 'Ἵ': 'Ι', 'Ἷ': 'Ι', 'Ὶ': 'Ι', 'Ί': 'Ι', 'Ῐ': 'Ι', 'Ῑ': 'Ι',
    'ό': 'ο', 'ὸ': 'ο', 'ὀ': 'ο', 'ὄ': 'ο', 'ὂ': 'ο', 'ὁ': 'ο', 'ὅ': 'ο',
    'Ό': 'Ο', 'Ὀ': 'Ο', 'Ὄ': 'Ο', 'Ὂ': 'Ο', 'Ὁ': 'Ο', 'Ὅ': 'Ο', 'Ὸ': 'Ο', 'Ό': 'Ο',
    'ύ': 'υ', 'ὺ': 'υ', 'ῦ': 'υ', 'ὐ': 'υ', 'ὔ': 'υ', 'ὒ': 'υ', 'ὖ': 'υ', 'ὑ': 'υ', 'ὕ': 'υ', 'ὗ': 'υ', 'ῢ': 'υ', 'ΰ': 'υ', 'ῦ': 'υ',
    'Ύ': 'Υ', 'Ὑ': 'Υ', 'Ὕ': 'Υ', 'Ὓ': 'Υ', 'Ὗ': 'Υ', 'Ὺ': 'Υ', 'Ύ': 'Υ', 'Ῠ': 'Υ', 'Ῡ': 'Υ',
    'ώ': 'ω', 'ὼ': 'ω', 'ῶ': 'ω', 'ὠ': 'ω', 'ὤ': 'ω', 'ὢ': 'ω', 'ὦ': 'ω', 'ὡ': 'ω', 'ὥ': 'ω', 'ὧ': 'ω', 'ᾠ': 'ω',
    'ᾡ': 'ω', 'ᾢ': 'ω', 'ᾣ': 'ω', 'ᾤ': 'ω', 'ᾥ': 'ω', 'ᾦ': 'ω', 'ᾧ': 'ω', 'ῲ': 'ω', 'ῴ': 'ω', 'ῶ': 'ω', 'ῷ': 'ω',
    'Ώ': 'Ω', 'Ὠ': 'Ω', 'Ὤ': 'Ω', 'Ὢ': 'Ω', 'Ὦ': 'Ω', 'Ὡ': 'Ω', 'Ὥ': 'Ω', 'Ὧ': 'Ω', 'ᾨ': 'Ω', 'ᾩ': 'Ω', 'ᾪ': 'Ω',
    'ᾫ': 'Ω', 'ᾬ': 'Ω', 'ᾭ': 'Ω', 'ᾮ': 'Ω', 'ᾯ': 'Ω', 'Ὼ': 'Ω', 'Ώ': 'Ω', 'ῼ': 'Ω',
    'ῤ': 'ρ', 'ῥ': 'ρ', 'Ῥ': 'Ρ'
}
GREEK_ACCENT_RE = re.compile('|'.join(GREEK_ACCENT_MAP.keys()))
GREEK_ALPHABET = "ΑΒΓΔΕΖΗΘΙΚΛΜΝΞΟΠΡΣΤΥΦΧΨΩ"
GREEK_ALPHABET_LOWER = "αβγδεζηθικλμνξοπρστυφχψω"

def strip_accents(text):
    if not text:
        return ""
    return GREEK_ACCENT_RE.sub(lambda m: GREEK_ACCENT_MAP[m.group(0)], text)

# --- 2. TEI-to-HTML Renderer Function ---
XML_FILE = "../GRC_misc/aeschylus-gemini.dindorf.lexicon.xml"
NAMESPACES = {'tei': 'http://www.tei-c.org/ns/1.0', 'xml': 'http://www.w3.org/XML/1998/namespace'}

TAG_MAP = {
    "form": "div", "orth": "span", "sense": "section", "def": "p",
    "cit": "blockquote", "bibl": "cite", "quote": "span", "note": "p",
    "xr": "p", "ref": "a", "foreign": "i", "usg": "span",
    "l": "div", "pb": "hr", "head": "h4"
}
CSS_MAP = {
    "form": "form", "orth": "orth", "sense": "sense", "def": "definition",
    "cit": "citation", "bibl": "bibl", "quote": "quote", "note": "note",
    "xr": "cross-reference", "ref": "ref", "foreign": "foreign", "usg": "usage",
    "l": "line", "pb": "pb", "head": "head"
}

def render_element(el):
    tag_name = el.tag.replace(f"{{{NAMESPACES['tei']}}}", "")
    html_tag = TAG_MAP.get(tag_name, "span")
    css_class = CSS_MAP.get(tag_name, tag_name)
    parts = []
    parts.append(f'<{html_tag} class="{css_class}"')
    
    if tag_name == "quote":
        lang = el.get('{http://www.w3.org/XML/1998/namespace}lang', 'grc')
        parts.append(f' lang="{lang}"')
    if tag_name == "ref":
        target = el.get('target', '#').lstrip('#')
        parts.append(f' href="#{target}"')
    if tag_name == "pb":
        n = el.get('n', '')
        parts.append(f' data-page-number="{n}"')
    
    parts.append('>')
    if el.text:
        parts.append(html.escape(el.text))
    for child in el:
        parts.append(render_element(child))
    parts.append(f'</{html_tag}>')
    if el.tail:
        parts.append(html.escape(el.tail))
    return "".join(parts)

def get_headword(entry):
    if entry is None:
        return "???"
    orth_el = entry.find('.//tei:orth', NAMESPACES)
    return orth_el.text if orth_el is not None and orth_el.text is not None else "???"

# --- 3. CSS Definition (Unchanged) ---

CSS = """
<style>
    body {
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
        line-height: 1.6;
        background-color: #fdfdfd;
        color: #333;
        margin: 0;
        padding: 0;
    }
    .main-container {
        display: grid;
        grid-template-columns: 300px 1fr;
        height: 100vh;
    }
    
    /* --- Table of Contents --- */
    #toc-container {
        border-right: 1px solid #ddd;
        padding: 20px;
        overflow-y: auto;
        height: 100vh;
        position: sticky;
        top: 0;
        background-color: #f9f9f9;
    }
    #toc-container h1 {
        font-size: 1.2em;
        margin-top: 0;
    }
    #search-bar {
        width: 100%;
        padding: 8px;
        margin-bottom: 15px;
        border: 1px solid #ccc;
        border-radius: 4px;
        box-sizing: border-box; /* Important */
    }
    
    /* --- A-Z Bar --- */
    #az-bar {
        display: flex;
        flex-wrap: wrap;
        justify-content: center;
        margin-bottom: 15px;
        padding-bottom: 10px;
        border-bottom: 1px solid #eee;
    }
    .az-letter {
        font-size: 1.1em;
        font-weight: bold;
        color: #0066cc;
        cursor: pointer;
        padding: 2px 5px;
        border-radius: 4px;
        transition: background-color 0.2s;
        text-decoration: none;
    }
    .az-letter:hover {
        background-color: #e6f0fa;
        text-decoration: none;
    }
    
    /* --- "Show All" Button --- */
    #show-all-btn {
        display: none; /* Hidden by default */
        font-weight: bold;
        color: #0066cc;
        cursor: pointer;
        margin-bottom: 15px;
        padding: 5px;
        border: 1px solid #ddd;
        border-radius: 4px;
        text-align: center;
        background-color: #fff;
    }
    #show-all-btn:hover {
        background-color: #e6f0fa;
    }

    /* --- Browse TOC (30 Chunks) --- */
    #toc-browse details {
        margin-bottom: 5px;
    }
    #toc-browse summary {
        cursor: pointer;
        font-weight: bold;
        padding: 5px;
        border-radius: 4px;
        transition: background-color 0.2s;
    }
    #toc-browse summary:hover {
        background-color: #eee;
    }
    .toc-sub-item {
        margin-left: 20px;
    }
    .toc-sub-item a {
        display: block;
        padding: 3px 5px;
        text-decoration: none;
        color: #0066cc;
        border-radius: 4px;
        transition: background-color 0.2s;
    }
    .toc-sub-item a:hover {
        background-color: #e6f0fa;
        text-decoration: underline;
    }
    
    /* --- Dynamic Drill-Down TOC --- */
    #toc-dynamic {
        display: none; /* Hidden by default */
    }
    .toc-dynamic-item {
        display: block;
        padding: 2px 5px;
        text-decoration: none;
        color: #0066cc;
        border-radius: 4px;
        transition: background-color 0.2s;
        white-space: nowrap;
        overflow: hidden;
        text-overflow: ellipsis;
    }
    .toc-dynamic-item:hover {
        background-color: #e6f0fa;
    }

    /* --- Content --- */
    #content-container {
        padding: 20px 40px;
        overflow-y: auto;
        height: 100vh;
        scroll-behavior: smooth; /* Smooth scrolling for TOC links */
    }
    .chunk-anchor {
        display: block;
        height: 10px;
        margin-top: -10px;
    }
    .lex-entry {
        border-bottom: 1px solid #eee;
        padding: 15px 0;
    }
    .lex-entry:target {
        /* Highlight entry when linked from TOC */
        animation: highlight 1.5s ease-out;
    }
    @keyframes highlight {
        0% { background-color: #ffffcc; }
        100% { background-color: transparent; }
    }
    .lex-entry:last-child {
        border-bottom: none;
    }
    
    /* --- TEI Element Styles (Unchanged) --- */
    .form { margin-bottom: 10px; }
    .orth {
        font-size: 1.5em;
        font-weight: bold;
        color: #1a1a1a;
        font-family: "Georgia", "Times New Roman", serif;
    }
    .sense {
        margin-left: 15px;
        padding-left: 10px;
        border-left: 2px solid #e0e0e0;
        margin-top: 10px;
    }
    .definition {
        font-style: italic;
        color: #555;
        margin: 5px 0;
    }
    .citation {
        margin-left: 20px;
        padding: 5px 10px;
        font-family: "Georgia", "Times New Roman", serif;
        border-left: 2px solid #f0f0f0;
        margin-top: 5px;
        margin-bottom: 5px;
    }
    cite.bibl {
        font-style: normal;
        font-weight: bold;
        color: #333;
        margin-right: 10px;
    }
    span.quote { color: #005588; }
    span.quote[lang="lat"] { color: #663300; }
    p.note {
        font-size: 0.9em;
        color: #666;
        background-color: #f7f7f7;
        border: 1px solid #eaeaea;
        border-radius: 4px;
        padding: 8px;
        margin: 5px 0;
    }
    p.cross-reference { font-size: 0.9em; color: #0066cc; }
    a.ref {
        text-decoration: none;
        color: #0066cc;
        border-bottom: 1px dotted #0066cc;
    }
    a.ref:hover { text-decoration: underline; }
    i.foreign { font-family: "Georgia", "Times New Roman", serif; }
    hr.pb {
        border: 0;
        height: 1px;
        background: #ccc;
        margin: 20px 0;
    }
    hr.pb::after {
        content: "Page " attr(data-page-number);
        display: block;
        text-align: center;
        color: #999;
        font-size: 0.8em;
    }
</style>
"""

# --- 4. ***UPDATED*** JavaScript Definition ---

JAVASCRIPT_TEMPLATE = """
<script>
    document.addEventListener('DOMContentLoaded', () => {
    
        // --- Injected Data from Python ---
        const headwordMap = {headword_map_json};

        // --- Accent Stripping ---
        const JS_ACCENT_MAP = {
            'ά': 'α', 'ὰ': 'α', 'ᾶ': 'α', 'ἀ': 'α', 'ἄ': 'α', 'ἂ': 'α', 'ἆ': 'α', 'ἁ': 'α', 'ἅ': 'α', 'ἇ': 'α', 'ᾀ': 'α',
            'ᾁ': 'α', 'ᾂ': 'α', 'ᾃ': 'α', 'ᾄ': 'α', 'ᾅ': 'α', 'ᾆ': 'α', 'ᾇ': 'α', 'ᾲ': 'α', 'ᾴ': 'α', 'ᾶ': 'α', 'ᾷ': 'α',
            'Ά': 'Α', 'Ἀ': 'Α', 'Ἄ': 'Α', 'Ἂ': 'Α', 'Ἆ': 'Α', 'Ἁ': 'Α', 'Ἅ': 'Α', 'Ἇ': 'Α', 'ᾈ': 'Α', 'ᾉ': 'Α', 'ᾊ': 'Α',
            'ᾋ': 'Α', 'ᾌ': 'Α', 'ᾍ': 'Α', 'ᾎ': 'Α', 'ᾏ': 'Α', 'Ὰ': 'Α', 'Ά': 'Α', 'Ᾰ': 'Α', 'Ᾱ': 'Α', 'ᾼ': 'Α',
            'έ': 'ε', 'ὲ': 'ε', 'ἐ': 'ε', 'ἔ': 'ε', 'ἒ': 'ε', 'ἑ': 'ε', 'ἕ': 'ε',
            'Έ': 'Ε', 'Ἐ': 'Ε', 'Ἔ': 'Ε', 'Ἒ': 'Ε', 'Ἑ': 'Ε', 'Ἕ': 'Ε', 'Ὲ': 'Ε', 'Έ': 'Ε',
            'ή': 'η', 'ὴ': 'η', 'ῆ': 'η', 'ἠ': 'η', 'ἤ': 'η', 'ἢ': 'η', 'ἦ': 'η', 'ἡ': 'η', 'ἥ': 'η', 'ἧ': 'η', 'ᾐ': 'η',
            'ᾑ': 'η', 'ᾒ': 'η', 'ᾓ': 'η', 'ᾔ': 'η', 'ᾕ': 'η', 'ᾖ': 'η', 'ᾗ': 'η', 'ῂ': 'η', 'ῄ': 'η', 'ῆ': 'η', 'ῇ': 'η',
            'Ή': 'Η', 'Ἠ': 'Η', 'Ἤ': 'Η', 'Ἢ': 'Η', 'Ἦ': 'Η', 'Ἡ': 'Η', 'Ἥ': 'Η', 'Ἧ': 'Η', 'ᾘ': 'Η', 'ᾙ': 'Η', 'ᾚ': 'Η',
            'ᾛ': 'Η', 'ᾜ': 'Η', 'ᾝ': 'Η', 'ᾞ': 'Η', 'ᾟ': 'Η', 'Ὴ': 'Η', 'Ή': 'Η', 'ῌ': 'Η',
            'ί': 'ι', 'ὶ': 'ι', 'ῖ': 'ι', 'ἰ': 'ι', 'ἴ': 'ι', 'ἲ': 'ι', 'ἶ': 'ι', 'ἱ': 'ι', 'ἵ': 'ι', 'ἷ': 'ι', 'ῒ': 'ι', 'ΐ': 'ι', 'ῖ': 'ι',
            'Ί': 'Ι', 'Ἰ': 'Ι', 'Ἴ': 'Ι', 'Ἲ': 'Ι', 'Ἶ': 'Ι', 'Ἱ': 'Ι', 'Ἵ': 'Ι', 'Ἷ': 'Ι', 'Ὶ': 'Ι', 'Ί': 'Ι', 'Ῐ': 'Ι', 'Ῑ': 'Ι',
            'ό': 'ο', 'ὸ': 'ο', 'ὀ': 'ο', 'ὄ': 'ο', 'ὂ': 'ο', 'ὁ': 'ο', 'ὅ': 'ο',
            'Ό': 'Ο', 'Ὀ': 'Ο', 'Ὄ': 'Ο', 'Ὂ': 'Ο', 'Ὁ': 'Ο', 'Ὅ': 'Ο', 'Ὸ': 'Ο', 'Ό': 'Ο',
            'ύ': 'υ', 'ὺ': 'υ', 'ῦ': 'υ', 'ὐ': 'υ', 'ὔ': 'υ', 'ὒ': 'υ', 'ὖ': 'υ', 'ὑ': 'υ', 'ὕ': 'υ', 'ὗ': 'υ', 'ῢ': 'υ', 'ΰ': 'υ', 'ῦ': 'υ',
            'Ύ': 'Υ', 'Ὑ': 'Υ', 'Ὕ': 'Υ', 'Ὓ': 'Υ', 'Ὗ': 'Υ', 'Ὺ': 'Υ', 'Ύ': 'Υ', 'Ῠ': 'Υ', 'Ῡ': 'Υ',
            'ώ': 'ω', 'ὼ': 'ω', 'ῶ': 'ω', 'ὠ': 'ω', 'ὤ': 'ω', 'ὢ': 'ω', 'ὦ': 'ω', 'ὡ': 'ω', 'ὥ': 'ω', 'ὧ': 'ω', 'ᾠ': 'ω',
            'ᾡ': 'ω', 'ᾢ': 'ω', 'ᾣ': 'ω', 'ᾤ': 'ω', 'ᾥ': 'ω', 'ᾦ': 'ω', 'ᾧ': 'ω', 'ῲ': 'ω', 'ῴ': 'ω', 'ῶ': 'ω', 'ῷ': 'ω',
            'Ώ': 'Ω', 'Ὠ': 'Ω', 'Ὤ': 'Ω', 'Ὢ': 'Ω', 'Ὦ': 'Ω', 'Ὡ': 'Ω', 'Ὥ': 'Ω', 'Ὧ': 'Ω', 'ᾨ': 'Ω', 'ᾩ': 'Ω', 'ᾪ': 'Ω',
            'ᾫ': 'Ω', 'ᾬ': 'Ω', 'ᾭ': 'Ω', 'ᾮ': 'Ω', 'ᾯ': 'Ω', 'Ὼ': 'Ω', 'Ώ': 'Ω', 'ῼ': 'Ω',
            'ῤ': 'ρ', 'ῥ': 'ρ', 'Ῥ': 'Ρ'
        };
        const JS_ACCENT_REGEX = new RegExp(Object.keys(JS_ACCENT_MAP).join('|'), 'g');
        function stripAccentsJS(s) {
            return s.replace(JS_ACCENT_REGEX, (matched) => JS_ACCENT_MAP[matched]);
        }

        // --- Global Element Selectors ---
        const searchBar = document.getElementById('search-bar');
        const allEntries = document.querySelectorAll('.lex-entry');
        const tocBrowse = document.getElementById('toc-browse');
        const tocDynamic = document.getElementById('toc-dynamic');
        const azBar = document.getElementById('az-bar');
        const showAllBtn = document.getElementById('show-all-btn');
        
        // --- State Variables ---
        let currentView = 'browse';
        let currentLetter = '';

        // --- Unified Filter Function ---
        function filterEntries() {
            let query = stripAccentsJS(searchBar.value).toLowerCase();
            let letterFilter = (currentView === 'letter') ? currentLetter : null;
            
            let visibleChunkIds = new Set();
            
            // 1. FILTER MAIN CONTENT
            for (let entry of allEntries) {
                const key = entry.getAttribute('data-search-key');
                const letter = entry.getAttribute('data-letter');
                
                const letterMatch = (!letterFilter) || (letter === letterFilter);
                const queryMatch = key.startsWith(query);
                
                if (letterMatch && queryMatch) {
                    entry.style.display = "";
                    if (currentView === 'browse' && query.length > 0) {
                        let anchor = entry.previousElementSibling;
                        while (anchor && !anchor.classList.contains('chunk-anchor')) {
                            anchor = anchor.previousElementSibling;
                        }
                        if (anchor) { visibleChunkIds.add(anchor.id); }
                    }
                } else {
                    entry.style.display = "none";
                }
            }
            
            // 2. FILTER TOCs
            if (currentView === 'browse') {
                const tocLinks = tocBrowse.querySelectorAll('.toc-sub-item');
                const detailsGroups = tocBrowse.querySelectorAll('details');
                
                if (query.length > 0) {
                    tocLinks.forEach(link => {
                        let a = link.querySelector('a');
                        let chunkId = a.getAttribute('href').substring(1);
                        if (visibleChunkIds.has(chunkId)) {
                            link.style.display = 'block';
                            let parentDetails = link.closest('details');
                            if (parentDetails) {
                                parentDetails.open = true;
                                parentDetails.style.display = 'block';
                            }
                        } else {
                            link.style.display = 'none';
                        }
                    });
                    detailsGroups.forEach(details => {
                        if (!details.querySelector('.toc-sub-item[style*="block"]')) {
                            details.style.display = 'none';
                        }
                    });
                } else {
                    tocLinks.forEach(link => link.style.display = 'block');
                    detailsGroups.forEach(details => {
                        details.style.display = 'block';
                        details.open = false;
                    });
                }
            } else if (currentView === 'letter') {
                const letterTocLinks = tocDynamic.querySelectorAll('.toc-dynamic-item');
                for (let link of letterTocLinks) {
                    let key = link.getAttribute('data-search-key');
                    if (key.startsWith(query)) {
                        link.style.display = 'block';
                    } else {
                        link.style.display = 'none';
                    }
                }
            }
        }

        // --- Navigation Functions ---
        
        function buildDynamicTOC(letter) {
            const headwords = headwordMap[letter] || [];
            let tocHtml = headwords.map(([headword, entryId]) => {
                let searchKey = stripAccentsJS(headword).toLowerCase();
                return `<a class="toc-dynamic-item" href="#${entryId}" data-search-key="${searchKey}">${headword}</a>`;
            }).join('');
            tocDynamic.innerHTML = tocHtml;
        }

        function showLetter(letter) {
            currentView = 'letter';
            currentLetter = letter;
            
            buildDynamicTOC(letter);
            
            tocBrowse.style.display = 'none';
            tocDynamic.style.display = 'block';
            azBar.style.display = 'none';
            showAllBtn.style.display = 'block';
            
            searchBar.value = "";
            filterEntries(); 
        }

        function showAll() {
            currentView = 'browse';
            currentLetter = '';
            
            tocBrowse.style.display = 'block';
            tocDynamic.style.display = 'none';
            azBar.style.display = 'flex';
            showAllBtn.style.display = 'none';
            
            searchBar.value = "";
            filterEntries(); 
        }

        // --- Attach Event Listeners ---
        
        if (searchBar) {
            searchBar.addEventListener('keyup', filterEntries);
        }
        
        if (showAllBtn) {
            showAllBtn.addEventListener('click', showAll);
        }
        
        if (azBar) {
            azBar.querySelectorAll('.az-letter').forEach(letterBtn => {
                letterBtn.addEventListener('click', (e) => {
                    e.preventDefault();
                    let letter = letterBtn.getAttribute('data-letter');
                    showLetter(letter);
                    document.getElementById('toc-container').scrollTop = 0;
                });
            });
        }
    
        // --- *** NEW: Handle URL Parameters *** ---
        function checkURLParams() {
            const urlParams = new URLSearchParams(window.location.search);
            // We'll use 'q' for a search query
            const query = urlParams.get('q');
            // We'll use 'letter' for the A-Z drilldown
            const letter = urlParams.get('letter');
            
            if (letter && headwordMap[letter]) {
                // If a letter is specified, go to that letter's drill-down view
                showLetter(letter);
                
                // If a query is ALSO specified, apply it
                if (query && query.trim() !== '') {
                    searchBar.value = query;
                    filterEntries();
                }
                
                // Scroll to the first match if there is one
                const firstEntry = document.querySelector(`.lex-entry[data-letter="${letter}"]`);
                if(firstEntry) {
                    firstEntry.scrollIntoView({ behavior: 'auto', block: 'start' });
                }

            } else if (query && query.trim() !== '') {
                // If only a query is specified, stay in browse mode and search
                searchBar.value = query;
                filterEntries();
                
                // Scroll to the first match
                const firstMatch = document.querySelector('.lex-entry[style*="display:"]');
                if (firstMatch) {
                    firstMatch.scrollIntoView({ behavior: 'auto', block: 'start' });
                }
            }
        }
        
        // Run the function on page load
        checkURLParams();
        
    }); // <-- Close DOMContentLoaded
</script>
"""

# --- 5. Main Parsing and HTML Building (Unchanged) ---

def create_html_dictionary():
    try:
        print(f"Starting to parse {XML_FILE}...")
        parser = ET.XMLParser(remove_blank_text=True)
        tree = ET.parse(XML_FILE, parser)
        root = tree.getroot()
        
        entries = root.findall('.//tei:entry', NAMESPACES)
        total_entries = len(entries)
        print(f"Found {total_entries} entries.")
        if total_entries == 0:
            print("No entries found. Check XML file path and namespaces.")
            return

        html_parts = []
        
        # --- Build HTML Head ---
        html_parts.append('<!DOCTYPE html>\n<html lang="en">\n<head>')
        html_parts.append('<meta charset="UTF-8">')
        html_parts.append('<meta name="viewport" content="width=device-width, initial-scale=1.0">')
        html_parts.append('<title>Lexicon Aeschyleum (Dindorf)</title>')
        html_parts.append(CSS)
        html_parts.append('</head>\n<body>\n<div class="main-container">')

        # --- Data for TOCs ---
        headword_map = {letter: [] for letter in GREEK_ALPHABET_LOWER}
        entry_data_for_build = [] 

        print("Analyzing headwords...")
        for idx, entry in enumerate(entries):
            entry_id = entry.get('{http://www.w3.org/XML/1998/namespace}id', f'entry-{idx}')
            headword = get_headword(entry)
            search_key = strip_accents(headword).lower()
            first_letter = search_key[0] if search_key else ''
            
            if first_letter in headword_map:
                headword_map[first_letter].append([headword, entry_id])
            
            entry_data_for_build.append({
                "id": entry_id,
                "headword": headword,
                "search_key": search_key,
                "first_letter": first_letter,
                "element": entry
            })

        # --- Build TOC (Left Column) ---
        print("Building Table of Contents...")
        toc_parts = [
            '<aside id="toc-container">',
            '<h1>Lexicon Aeschyleum</h1>',
            '<nav id="az-bar">'
        ]
        for letter_upper, letter_lower in zip(GREEK_ALPHABET, GREEK_ALPHABET_LOWER):
            toc_parts.append(f'<a href="#" class="az-letter" data-letter="{letter_lower}">{letter_upper}</a>')
        toc_parts.append('</nav>')
        toc_parts.append('<div id="show-all-btn">« Show All Entries (Browse)</div>')
        toc_parts.append('<input type="text" id="search-bar" placeholder="Search (unaccented Greek)...">')
        toc_parts.append('<nav id="toc-browse">')
        
        num_top_chunks = 10
        num_sub_chunks = 3
        sub_chunk_size = math.ceil(total_entries / (num_top_chunks * num_sub_chunks))
        top_chunk_size = sub_chunk_size * num_sub_chunks
        
        for i in range(num_top_chunks):
            top_start_idx = i * top_chunk_size
            top_end_idx = min((i + 1) * top_chunk_size - 1, total_entries - 1)
            
            if top_start_idx > total_entries - 1:
                break
            
            first_headword = entry_data_for_build[top_start_idx]["headword"]
            last_headword = entry_data_for_build[top_end_idx]["headword"]
            
            toc_parts.append(f'<details><summary>{first_headword} — {last_headword}</summary>')
            
            for j in range(num_sub_chunks):
                sub_start_idx = top_start_idx + (j * sub_chunk_size)
                if sub_start_idx >= total_entries:
                    break
                sub_end_idx = min(sub_start_idx + sub_chunk_size - 1, top_end_idx)
                
                chunk_id = f"chunk-{i}-{j}"
                first_sub_headword = entry_data_for_build[sub_start_idx]["headword"]
                last_sub_headword = entry_data_for_build[sub_end_idx]["headword"]
                
                toc_parts.append(f'<div class="toc-sub-item"><a href="#{chunk_id}">{first_sub_headword} — {last_sub_headword}</a></div>')
            
            toc_parts.append('</details>')
            
        toc_parts.append('</nav>')
        toc_parts.append('<nav id="toc-dynamic"></nav>')
        toc_parts.append('</aside>')
        html_parts.append("".join(toc_parts))
        
        # --- Build Content (Right Column) ---
        print("Building main content...")
        content_parts = ['<main id="content-container">']
        
        for idx, entry_data in enumerate(entry_data_for_build):
            if idx % sub_chunk_size == 0:
                chunk_num = idx // sub_chunk_size
                i = chunk_num // num_sub_chunks
                j = chunk_num % num_sub_chunks
                chunk_id = f"chunk-{i}-{j}"
                content_parts.append(f'<div class="chunk-anchor" id="{chunk_id}"></div>')
            
            entry = entry_data["element"]
            
            content_parts.append(f'<article class="lex-entry" id="{entry_data["id"]}" data-search-key="{entry_data["search_key"]}" data-letter="{entry_data["first_letter"]}">')
            
            for child in entry:
                content_parts.append(render_element(child))
                
            content_parts.append('</article>')
        
        content_parts.append('</main>')
        html_parts.append("".join(content_parts))
        
        # --- Close tags and add JavaScript ---
        html_parts.append('</div>')
        
        js_code = JAVASCRIPT_TEMPLATE.replace(
            '{headword_map_json}', 
            json.dumps(headword_map, ensure_ascii=False)
        )
        html_parts.append(js_code)
        
        html_parts.append('</body>\n</html>')
        
        # --- Write to file ---
        output_filename = "aeschylus_lexicon.html"
        print(f"Writing HTML to {output_filename}...")
        final_html = "".join(html_parts)
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(final_html)
            
        print(f"Successfully created {output_filename}.")
        print("You can now open this file in your browser.")
        
    except FileNotFoundError:
        print(f"Error: The file '{XML_FILE}' was not found.")
        print("Please make sure it's in the same directory as this notebook.")
    except ET.ParseError as e:
        print(f"Error parsing XML: {e}")
        print("There may be a validation error in your XML file.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Execute the main function
create_html_dictionary()

Starting to parse ../GRC_misc/aeschylus-gemini.dindorf.lexicon.xml...
Found 7488 entries.
Analyzing headwords...
Building Table of Contents...
Building main content...
Writing HTML to aeschylus_lexicon.html...
Successfully created aeschylus_lexicon.html.
You can now open this file in your browser.
